# Импорт библиотек

In [16]:
import numpy as np
import pandas as pd
import torch
import torchvision
from torchvision import transforms, datasets, models
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2
import cv2
import os
import glob
import xml.etree.ElementTree as et
from collections import defaultdict
import json
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import yaml
import random
import shutil

print("Все библиотеки успешно загружены!")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\DELL\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Все библиотеки успешно загружены!


# Загрузка данных в DataLoader

In [17]:
# Определение путей к данным
BASE_PATH = r'C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2'

# Пути к изображениям и аннотациям
images_dir = os.path.join(BASE_PATH, 'images')
annotations_dir = os.path.join(BASE_PATH, 'annotations')

# Создание директории для результатов
output_dir = os.path.join(BASE_PATH, 'output')
os.makedirs(output_dir, exist_ok=True)

# Проверка существования директорий
print("=" * 60)
print("ПРОВЕРКА ПУТЕЙ К ДАННЫМ")
print("=" * 60)
print(f"Базовая директория: {BASE_PATH}")
print(f"Директория с изображениями: {images_dir}")
print(f"  Существует: {os.path.exists(images_dir)}")
print(f"Директория с аннотациями: {annotations_dir}")
print(f"  Существует: {os.path.exists(annotations_dir)}")

if os.path.exists(images_dir):
    img_count = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"\n✓ Изображений найдено: {img_count}")

if os.path.exists(annotations_dir):
    ann_count = len([f for f in os.listdir(annotations_dir) if f.lower().endswith('.xml')])
    print(f"✓ Аннотаций найдено: {ann_count}")

print(f"\nДиректория результатов: {output_dir}")

ПРОВЕРКА ПУТЕЙ К ДАННЫМ
Базовая директория: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2
Директория с изображениями: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\images
  Существует: True
Директория с аннотациями: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\annotations
  Существует: True

✓ Изображений найдено: 853
✓ Аннотаций найдено: 853

Директория результатов: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output


In [18]:
class FaceMaskDataset(torch.utils.data.Dataset):
    """
    Датасет для детектирования медицинских масок.
    
    Классы:
    - 0: background (фон)
    - 1: without_mask (без маски)
    - 2: with_mask (в маске)
    - 3: mask_weared_incorrect (маска надета неправильно)
    """
    
    def __init__(self, images_dir, annotation_dir, width=480, height=480, transforms=None):
        self.transforms = transforms
        self.images_dir = images_dir
        self.annotation_dir = annotation_dir
        self.height = height
        self.width = width
        
        # Получаем списки файлов
        all_imgs = sorted([f for f in os.listdir(images_dir) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        all_anns = sorted([f for f in os.listdir(annotation_dir) 
                          if f.lower().endswith('.xml')])
        
        # Находим файлы, для которых есть и изображение, и аннотация
        img_stems = {os.path.splitext(f)[0] for f in all_imgs}
        ann_stems = {os.path.splitext(f)[0] for f in all_anns}
        common_stems = sorted(img_stems & ann_stems)
        
        # Проверяем целостность файлов
        self.imgs = []
        self.annotate = []
        broken_count = 0
        
        for stem in common_stems:
            # Ищем файлы с разными расширениями
            img_file = None
            for ext in ['.png', '.jpg', '.jpeg']:
                candidate = stem + ext
                if candidate in all_imgs:
                    img_file = candidate
                    break
            
            ann_file = stem + '.xml'
            
            if img_file and ann_file in all_anns:
                # Проверяем, что изображение читается
                test_img = cv2.imread(os.path.join(images_dir, img_file))
                if test_img is not None:
                    self.imgs.append(img_file)
                    self.annotate.append(ann_file)
                else:
                    broken_count += 1
        
        if broken_count > 0:
            print(f"Пропущено поврежденных изображений: {broken_count}")
        
        print(f"Датасет создан: {len(self.imgs)} валидных изображений, размер: {width}x{height}")
        
        self.classes = ['background', 'without_mask', 'with_mask', 'mask_weared_incorrect']
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        image_path = os.path.join(self.images_dir, img_name)
        
        img = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        img_res = cv2.resize(img_rgb, (self.width, self.height), cv2.INTER_AREA)
        img_res /= 255.0
        
        # Загрузка аннотации
        annot_path = os.path.join(self.annotation_dir, self.annotate[idx])
        
        boxes = []
        labels = []
        
        try:
            tree = et.parse(annot_path)
            root = tree.getroot()
            
            wt = img.shape[1]
            ht = img.shape[0]
            
            for member in root.findall('object'):
                class_name = member.find('name').text
                if class_name in self.classes:
                    labels.append(self.classes.index(class_name))
                
                bbox = member.find('bndbox')
                xmin = float(bbox.find('xmin').text)
                xmax = float(bbox.find('xmax').text)
                ymin = float(bbox.find('ymin').text)
                ymax = float(bbox.find('ymax').text)
                
                xmin_corr = (xmin / wt) * self.width
                xmax_corr = (xmax / wt) * self.width
                ymin_corr = (ymin / ht) * self.height
                ymax_corr = (ymax / ht) * self.height
                
                boxes.append([xmin_corr, ymin_corr, xmax_corr, ymax_corr])
        except Exception as e:
            pass  # Пропускаем ошибки парсинга
        
        # Конвертация в тензоры
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
            iscrowd = torch.zeros(0, dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'area': area,
            'iscrowd': iscrowd,
            'image_id': torch.tensor([idx])
        }
        
        # Применение аугментаций
        if self.transforms:
            if len(boxes) > 0:
                try:
                    sample = self.transforms(
                        image=img_res,
                        bboxes=boxes.tolist(),
                        labels=labels.tolist()
                    )
                    img_res = sample['image']
                    target['boxes'] = torch.as_tensor(sample['bboxes'], dtype=torch.float32)
                    target['labels'] = torch.as_tensor(sample['labels'], dtype=torch.int64)
                except Exception:
                    sample = A.Compose([ToTensorV2(p=1.0)])(image=img_res)
                    img_res = sample['image']
            else:
                sample = A.Compose([ToTensorV2(p=1.0)])(image=img_res)
                img_res = sample['image']
        
        return img_res, target

In [19]:
def get_transform(train=True):
    """
    Аугментации для обучения и валидации.
    
    Обоснование:
    - HorizontalFlip: инвариантность к повороту лица
    - RandomBrightnessContrast: устойчивость к освещению
    - RandomGamma: адаптация к условиям съемки
    """
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.RandomGamma(p=0.2),
            ToTensorV2(p=1.0)
        ], bbox_params=A.BboxParams(
            format='pascal_voc',
            label_fields=['labels'],
            min_visibility=0.3
        ))
    else:
        return A.Compose([
            ToTensorV2(p=1.0)
        ], bbox_params=A.BboxParams(
            format='pascal_voc',
            label_fields=['labels']
        ))

def collate_fn(batch):
    """Объединение элементов батча для Faster R-CNN"""
    return tuple(zip(*batch))

In [ ]:
print("\n" + "=" * 60)
print("СОЗДАНИЕ ДАТАСЕТОВ")
print("=" * 60)

IMG_SIZE = 480
BATCH_SIZE = 4
VAL_SPLIT = 0.2

dataset_train_full = FaceMaskDataset(images_dir, annotations_dir, IMG_SIZE, IMG_SIZE, get_transform(train=True))
dataset_val_full = FaceMaskDataset(images_dir, annotations_dir, IMG_SIZE, IMG_SIZE, get_transform(train=False))

# Разделение
torch.manual_seed(42)
indices = torch.randperm(len(dataset_train_full)).tolist()
val_size = int(len(dataset_train_full) * VAL_SPLIT)

dataset_train = torch.utils.data.Subset(dataset_train_full, indices[:-val_size])
dataset_val = torch.utils.data.Subset(dataset_val_full, indices[-val_size:])

print(f"Train: {len(dataset_train)}, Val: {len(dataset_val)}")

# DataLoader'ы
data_loader_train = torch.utils.data.DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
data_loader_val = torch.utils.data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

print("DataLoader'ы готовы!")


СОЗДАНИЕ ДАТАСЕТОВ И DATALOADER'ОВ
Создание датасетов...
Датасет создан: 853 изображений, размер: 480x480
Датасет создан: 853 изображений, размер: 480x480


c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\albumentations\core\composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Обучающая выборка: 683 изображений
Валидационная выборка: 170 изображений
DataLoader'ы созданы успешно!


In [20]:
class FaceMaskDataset(torch.utils.data.Dataset):
    """
    Датасет для детектирования медицинских масок.
    
    Загружает изображения и соответствующие XML-аннотации.
    Возвращает данные в формате, совместимом с Faster R-CNN.
    
    Классы:
    - 0: background (фон)
    - 1: without_mask (без маски)
    - 2: with_mask (в маске)
    - 3: mask_weared_incorrect (маска надета неправильно)
    """
    
    def __init__(self, images_dir, annotation_dir, width=480, height=480, transforms=None):
        """
        Args:
            images_dir: путь к директории с изображениями
            annotation_dir: путь к директории с аннотациями XML
            width: целевая ширина изображения
            height: целевая высота изображения
            transforms: albumentations transforms
        """
        self.transforms = transforms
        self.images_dir = images_dir
        self.annotation_dir = annotation_dir
        self.height = height
        self.width = width
        
        # Получаем отсортированные списки файлов
        self.imgs = sorted([f for f in os.listdir(images_dir) 
                           if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.annotate = sorted([f for f in os.listdir(annotation_dir) 
                               if f.lower().endswith('.xml')])
        
        # Проверяем, что для каждого изображения есть аннотация и наоборот
        # Создаем списки только с совпадающими именами файлов
        img_stems = {os.path.splitext(f)[0] for f in self.imgs}
        ann_stems = {os.path.splitext(f)[0] for f in self.annotate}
        
        # Находим общие файлы
        common_stems = img_stems & ann_stems
        
        if len(common_stems) < len(img_stems) or len(common_stems) < len(ann_stems):
            print(f"Предупреждение: не все файлы имеют пару изображение-аннотация")
            print(f"  Изображений: {len(img_stems)}, Аннотаций: {len(ann_stems)}, Общих: {len(common_stems)}")
        
        # Фильтруем списки, оставляя только парные файлы
        self.imgs = sorted([f for f in self.imgs if os.path.splitext(f)[0] in common_stems])
        self.annotate = sorted([f for f in self.annotate if os.path.splitext(f)[0] in common_stems])
        
        # Проверяем, что все изображения загружаются
        valid_imgs = []
        valid_anns = []
        for img_file, ann_file in zip(self.imgs, self.annotate):
            img_path = os.path.join(images_dir, img_file)
            test_img = cv2.imread(img_path)
            if test_img is not None:
                valid_imgs.append(img_file)
                valid_anns.append(ann_file)
            else:
                print(f"Пропущено поврежденное изображение: {img_file}")
        
        self.imgs = valid_imgs
        self.annotate = valid_anns
        
        # Классы для детекции (0 - background зарезервирован для Faster R-CNN)
        self.classes = ['background', 'without_mask', 'with_mask', 'mask_weared_incorrect']
        
        print(f"Датасет создан: {len(self.imgs)} изображений, размер: {width}x{height}")
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        """
        Возвращает изображение и target словарь для одного элемента датасета.
        """
        # Загрузка изображения
        img_name = self.imgs[idx]
        image_path = os.path.join(self.images_dir, img_name)
        
        img = cv2.imread(image_path)
        if img is None:
            # Если изображение не загрузилось, пробуем следующее
            print(f"Пропуск изображения {img_name} (индекс {idx})")
            # Возвращаем следующий валидный элемент
            return self.__getitem__((idx + 1) % len(self))
        
        # Конвертация BGR -> RGB и нормализация
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
        img_res = cv2.resize(img_rgb, (self.width, self.height), cv2.INTER_AREA)
        img_res /= 255.0
        
        # Загрузка аннотации
        annot_filename = self.annotate[idx]
        annot_file_path = os.path.join(self.annotation_dir, annot_filename)
        
        boxes = []
        labels = []
        
        try:
            tree = et.parse(annot_file_path)
            root = tree.getroot()
            
            wt = img.shape[1]
            ht = img.shape[0]
            
            for member in root.findall('object'):
                class_name = member.find('name').text
                if class_name in self.classes:
                    labels.append(self.classes.index(class_name))
                else:
                    continue
                
                bbox = member.find('bndbox')
                xmin = float(bbox.find('xmin').text)
                xmax = float(bbox.find('xmax').text)
                ymin = float(bbox.find('ymin').text)
                ymax = float(bbox.find('ymax').text)
                
                xmin_corr = (xmin / wt) * self.width
                xmax_corr = (xmax / wt) * self.width
                ymin_corr = (ymin / ht) * self.height
                ymax_corr = (ymax / ht) * self.height
                
                boxes.append([xmin_corr, ymin_corr, xmax_corr, ymax_corr])
        except Exception as e:
            print(f"Ошибка при чтении аннотации {annot_filename}: {e}")
        
        # Конвертация в тензоры
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
            iscrowd = torch.zeros(0, dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'area': area,
            'iscrowd': iscrowd,
            'image_id': torch.tensor([idx])
        }
        
        # Применение аугментаций
        if self.transforms:
            if len(boxes) > 0:
                try:
                    sample = self.transforms(
                        image=img_res,
                        bboxes=boxes.tolist(),
                        labels=labels.tolist()
                    )
                    img_res = sample['image']
                    if len(sample['bboxes']) > 0:
                        target['boxes'] = torch.as_tensor(sample['bboxes'], dtype=torch.float32)
                        target['labels'] = torch.as_tensor(sample['labels'], dtype=torch.int64)
                    else:
                        target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
                        target['labels'] = torch.zeros(0, dtype=torch.int64)
                except Exception:
                    transform = A.Compose([ToTensorV2(p=1.0)])
                    sample = transform(image=img_res)
                    img_res = sample['image']
            else:
                transform = A.Compose([ToTensorV2(p=1.0)])
                sample = transform(image=img_res)
                img_res = sample['image']
        
        return img_res, target

# Обучение модели

In [ ]:
def plot_img_bbox(img, target, class_names=None, title="Пример"):
    """Визуализация изображения с bounding boxes"""
    if class_names is None:
        class_names = ['background', 'without_mask', 'with_mask', 'mask_weared_incorrect']
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    if isinstance(img, torch.Tensor):
        img = img.permute(1, 2, 0).cpu().numpy()
    
    ax.imshow(img)
    colors = ['gray', 'red', 'green', 'orange']
    
    for i, box in enumerate(target['boxes']):
        if isinstance(box, torch.Tensor):
            box = box.cpu().numpy()
        x, y, w, h = box[0], box[1], box[2]-box[0], box[3]-box[1]
        label_idx = int(target['labels'][i])
        label_name = class_names[label_idx]
        
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=colors[label_idx], facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y-5, label_name, color='white', fontsize=10, 
                bbox=dict(facecolor=colors[label_idx], alpha=0.8))
    
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Показываем 3 примера
print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ ПРИМЕРОВ")
print("=" * 60)

for i in range(3):
    idx = random.randint(0, len(dataset_val)-1)
    img, target = dataset_val[idx]
    plot_img_bbox(img, target, title=f"Пример {i+1} (изображение {idx})")

In [21]:
def train_one_epoch(model, data_loader, optimizer, device, epoch):
    """
    Обучение модели Faster R-CNN в течение одной эпохи.
    """
    model.train()
    total_loss = 0
    progress_bar = tqdm(data_loader, desc=f'Epoch {epoch}')
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Прямой проход
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Обратный проход
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        progress_bar.set_postfix({'loss': f'{losses.item():.4f}'})
    
    return total_loss / len(data_loader)

print("\n" + "=" * 60)
print("ОБУЧЕНИЕ FASTER R-CNN")
print("=" * 60)

# Настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# Создание модели
num_classes = 4
model_rcnn = get_faster_rcnn_model(num_classes)
model_rcnn.to(device)

# Оптимизатор и планировщик
params = [p for p in model_rcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Обучение
NUM_EPOCHS = 15
for epoch in range(1, NUM_EPOCHS + 1):
    avg_loss = train_one_epoch(model_rcnn, data_loader_train, optimizer, device, epoch)
    lr_scheduler.step()
    print(f"Epoch {epoch}/{NUM_EPOCHS} - Average Loss: {avg_loss:.4f}")

# Сохранение модели
torch.save(model_rcnn.state_dict(), os.path.join(output_dir, 'faster_rcnn_mask.pth'))
print(f"\nМодель Faster R-CNN сохранена в {output_dir}/faster_rcnn_mask.pth")


ОБУЧЕНИЕ FASTER R-CNN
Используемое устройство: cpu


Epoch 1: 100%|██████████| 171/171 [1:36:04<00:00, 33.71s/it, loss=0.1578]


Epoch 1/15 - Average Loss: 0.4472


Epoch 2: 100%|██████████| 171/171 [1:31:29<00:00, 32.10s/it, loss=0.3164]


Epoch 2/15 - Average Loss: 0.3088


...

Epoch 13/15 - Average Loss: 0.1722


Epoch 14: 100%|██████████| 171/171 [1:43:33<00:00, 36.33s/it, loss=0.1375]  


Epoch 14/15 - Average Loss: 0.1721


Epoch 15: 100%|██████████| 171/171 [1:41:37<00:00, 35.66s/it, loss=0.0582] 


Epoch 15/15 - Average Loss: 0.1720

Модель Faster R-CNN сохранена в C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output/faster_rcnn_mask.pth


In [22]:
def get_faster_rcnn_model(num_classes):
    """
    Faster R-CNN с ResNet-50-FPN.
    
    Обоснование выбора:
    1. Двухэтапный детектор - высокая точность
    2. ResNet-50-FPN - баланс скорости и качества
    3. Предобучение на COCO - хорошая инициализация
    4. FPN улучшает детекцию объектов разного размера
    """
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def get_yolo_model():
    """
    YOLOv8n - одноэтапный детектор.
    
    Обоснование выбора:
    1. Высокая скорость инференса
    2. Современная архитектура
    3. Встроенные аугментации (mosaic, mixup)
    4. Nano версия - быстрое обучение
    """
    return YOLO('yolov8n.pt')

In [23]:
def train_one_epoch(model, data_loader, optimizer, device, epoch):
    """Обучение Faster R-CNN в течение одной эпохи"""
    model.train()
    total_loss = 0
    pbar = tqdm(data_loader, desc=f'Epoch {epoch}')
    
    for images, targets in pbar:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        pbar.set_postfix({'loss': f'{losses.item():.4f}'})
    
    return total_loss / len(data_loader)

In [24]:
print("\n" + "=" * 60)
print("ОБУЧЕНИЕ FASTER R-CNN")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

model_rcnn = get_faster_rcnn_model(num_classes=4)
model_rcnn.to(device)

params = [p for p in model_rcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

NUM_EPOCHS = 15
for epoch in range(1, NUM_EPOCHS + 1):
    avg_loss = train_one_epoch(model_rcnn, data_loader_train, optimizer, device, epoch)
    lr_scheduler.step()
    print(f"Epoch {epoch}/{NUM_EPOCHS} - Loss: {avg_loss:.4f}")

torch.save(model_rcnn.state_dict(), os.path.join(output_dir, 'faster_rcnn_mask.pth'))
print(f"Модель сохранена: {output_dir}/faster_rcnn_mask.pth")


ОБУЧЕНИЕ FASTER R-CNN
Устройство: cpu


c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1: 100%|██████████| 171/171 [1:48:24<00:00, 38.04s/it, loss=0.5213]  


Epoch 1/15 - Loss: 0.4386


Epoch 2: 100%|██████████| 171/171 [1:36:27<00:00, 33.85s/it, loss=0.3479]


Epoch 2/15 - Loss: 0.3025


...

Epoch 13/15 - Loss: 0.1730


Epoch 14: 100%|██████████| 171/171 [1:41:00<00:00, 35.44s/it, loss=0.0881]


Epoch 14/15 - Loss: 0.1693


Epoch 15: 100%|██████████| 171/171 [1:50:15<00:00, 38.69s/it, loss=0.1190]  


Epoch 15/15 - Loss: 0.1701
Модель сохранена: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output/faster_rcnn_mask.pth


In [25]:
def compute_iou(box1, box2):
    """Intersection over Union между двумя боксами"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

def compute_ap(recalls, precisions):
    """Average Precision по 11 точкам"""
    ap = 0
    for t in np.linspace(0, 1, 11):
        p = np.max(precisions[recalls >= t]) if np.sum(recalls >= t) > 0 else 0
        ap += p / 11
    return ap

def evaluate_rcnn(model, data_loader, device, iou_threshold=0.5):
    """Оценка Faster R-CNN - вычисление mAP"""
    model.eval()
    all_preds = []
    all_targets = []
    
    print("Оценка Faster R-CNN...")
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc='Eval'):
            images = list(img.to(device) for img in images)
            preds = model(images)
            
            for pred in preds:
                if len(pred['boxes']) > 0:
                    keep = torchvision.ops.nms(pred['boxes'], pred['scores'], 0.3)
                    pred = {k: v[keep] for k, v in pred.items()}
                all_preds.append(pred)
            all_targets.extend(targets)
    
    class_names = ['without_mask', 'with_mask', 'mask_weared_incorrect']
    class_aps = []
    
    for class_id in range(1, 4):
        detections = []
        num_gt = 0
        
        for pred, target in zip(all_preds, all_targets):
            mask = pred['labels'] == class_id
            pboxes = pred['boxes'][mask].cpu().numpy()
            pscores = pred['scores'][mask].cpu().numpy()
            
            gmask = target['labels'] == class_id
            gtboxes = target['boxes'][gmask].cpu().numpy()
            num_gt += len(gtboxes)
            
            for box, score in zip(pboxes, pscores):
                detections.append((score, box, gtboxes.copy()))
        
        if num_gt == 0:
            class_aps.append(0.0)
            continue
        
        detections.sort(key=lambda x: x[0], reverse=True)
        tp = np.zeros(len(detections))
        fp = np.zeros(len(detections))
        gt_used = {}
        
        for i, (score, box, gtboxes) in enumerate(detections):
            best_iou, best_gt = 0, -1
            for j, gt in enumerate(gtboxes):
                iou = compute_iou(box, gt)
                if iou > best_iou:
                    best_iou, best_gt = iou, j
            
            if best_iou >= iou_threshold and best_gt not in gt_used:
                tp[i] = 1
                gt_used[best_gt] = True
            else:
                fp[i] = 1
        
        cum_tp = np.cumsum(tp)
        cum_fp = np.cumsum(fp)
        recalls = cum_tp / num_gt
        precisions = cum_tp / (cum_tp + cum_fp + 1e-6)
        class_aps.append(compute_ap(recalls, precisions))
    
    mAP = np.mean(class_aps)
    return mAP, class_aps, all_preds, all_targets

In [26]:
print("\n" + "=" * 60)
print("ОЦЕНКА FASTER R-CNN")
print("=" * 60)

mAP_rcnn, class_aps, preds_rcnn, targets_rcnn = evaluate_rcnn(model_rcnn, data_loader_val, device)

print(f"\n{'='*60}")
print("РЕЗУЛЬТАТЫ FASTER R-CNN")
print(f"{'='*60}")
print(f"mAP @ IoU=0.5: {mAP_rcnn:.4f}")
print("\nAP по классам:")
for name, ap in zip(['without_mask', 'with_mask', 'mask_weared_incorrect'], class_aps):
    print(f"  {name}: {ap:.4f}")


ОЦЕНКА FASTER R-CNN
Оценка Faster R-CNN...


Eval: 100%|██████████| 43/43 [10:53<00:00, 15.19s/it]



РЕЗУЛЬТАТЫ FASTER R-CNN
mAP @ IoU=0.5: 0.0909

AP по классам:
  without_mask: 0.0909
  with_mask: 0.0909
  mask_weared_incorrect: 0.0909


# Отрисовка изображений

Подготовка данных для YOLO

In [28]:
print("\n" + "=" * 60)
print("ПОДГОТОВКА ДАННЫХ ДЛЯ YOLO")
print("=" * 60)

yolo_dir = os.path.join(BASE_PATH, 'yolo_dataset')

# Создание структуры папок
for d in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(yolo_dir, d), exist_ok=True)

# Списки файлов
img_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))])
ann_files = sorted([f for f in os.listdir(annotations_dir) if f.lower().endswith('.xml')])

# Общие файлы
common = sorted(set(os.path.splitext(f)[0] for f in img_files) & set(os.path.splitext(f)[0] for f in ann_files))
print(f"Найдено пар изображение-аннотация: {len(common)}")

# Разделение
np.random.seed(42)
indices = np.random.permutation(len(common))
val_size = int(len(common) * 0.2)
train_idx = indices[:-val_size]
val_idx = indices[-val_size:]

class_map = {'without_mask': 0, 'with_mask': 1, 'mask_weared_incorrect': 2}

def process_split(indices, split_name, img_subdir, lbl_subdir):
    for i in tqdm(indices, desc=split_name):
        stem = common[i]
        # Копируем изображение
        for ext in ['.png', '.jpg', '.jpeg']:
            src = os.path.join(images_dir, stem + ext)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(yolo_dir, img_subdir, stem + ext))
                break
        
        # Конвертируем аннотацию
        ann_path = os.path.join(annotations_dir, stem + '.xml')
        if not os.path.exists(ann_path):
            continue
        
        img = cv2.imread(os.path.join(images_dir, stem + '.png'))
        if img is None:
            img = cv2.imread(os.path.join(images_dir, stem + '.jpg'))
        if img is None:
            continue
        h, w = img.shape[:2]
        
        tree = et.parse(ann_path)
        root = tree.getroot()
        
        with open(os.path.join(yolo_dir, lbl_subdir, stem + '.txt'), 'w') as f:
            for obj in root.findall('object'):
                name = obj.find('name').text
                if name in class_map:
                    bbox = obj.find('bndbox')
                    xmin = float(bbox.find('xmin').text)
                    ymin = float(bbox.find('ymin').text)
                    xmax = float(bbox.find('xmax').text)
                    ymax = float(bbox.find('ymax').text)
                    xc = ((xmin + xmax) / 2) / w
                    yc = ((ymin + ymax) / 2) / h
                    bw = (xmax - xmin) / w
                    bh = (ymax - ymin) / h
                    f.write(f"{class_map[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

process_split(train_idx, 'Train', 'images/train', 'labels/train')
process_split(val_idx, 'Val', 'images/val', 'labels/val')

# YAML конфиг
config = {
    'path': yolo_dir.replace('\\', '/'),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': ['without_mask', 'with_mask', 'mask_weared_incorrect']
}
with open(os.path.join(yolo_dir, 'dataset.yaml'), 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Данные YOLO подготовлены: {yolo_dir}")
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")


ПОДГОТОВКА ДАННЫХ ДЛЯ YOLO
Найдено пар изображение-аннотация: 853


Val: 100%|██████████| 170/170 [00:05<00:00, 30.79it/s]


Данные YOLO подготовлены: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\yolo_dataset
Train: 683, Val: 170


In [29]:
print("\n" + "=" * 60)
print("ОБУЧЕНИЕ YOLOv8")
print("=" * 60)

model_yolo = YOLO('yolov8n.pt')
print("YOLOv8n загружен")

results = model_yolo.train(
    data=os.path.join(yolo_dir, 'dataset.yaml'),
    epochs=50,
    imgsz=480,
    batch=8,
    name='mask_detection',
    project=output_dir,
    patience=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True
)

print(f"\nYOLO обучен! Результаты: {output_dir}/mask_detection")


ОБУЧЕНИЕ YOLOv8
YOLOv8n загружен
New https://pypi.org/project/ultralytics/8.4.46 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i5-7500T 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Public\Documents\Documents\\SF\CV\Projects\Project_2\yolo_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, m

In [30]:
print("\n" + "=" * 60)
print("ОЦЕНКА YOLOv8")
print("=" * 60)

metrics = model_yolo.val(data=os.path.join(yolo_dir, 'dataset.yaml'), split='val')

print(f"\n{'='*60}")
print("РЕЗУЛЬТАТЫ YOLOv8")
print(f"{'='*60}")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

if hasattr(metrics.box, 'ap'):
    for i, ap in enumerate(metrics.box.ap):
        print(f"  {['without_mask','with_mask','mask_weared_incorrect'][i]}: {ap:.4f}")


ОЦЕНКА YOLOv8
Ultralytics 8.4.45  Python-3.11.4 torch-2.11.0+cpu CPU (Intel Core i5-7500T 2.70GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 3.33.9 ms, read: 48.143.0 MB/s, size: 553.9 KB)
val: Scanning C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\yolo_dataset\labels\val.cache... 170 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 170/170  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.6s/it 39.3s3.3s
                   all        170        995      0.901      0.723      0.791      0.504
          without_mask         58        161      0.817      0.658       0.72       0.46
             with_mask        153        809      0.941      0.845      0.917      0.586
 mask_weared_incorrect         16         25      0.943      0.665      0.735      0.466
Speed: 5.3ms preprocess, 187.3ms inference, 0.0ms loss, 3.0ms post

Визуализация предсказаний Faster R-CNN

In [32]:
print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ ПРЕДСКАЗАНИЙ FASTER R-CNN")
print("=" * 60)

def visualize_rcnn_predictions(model_rcnn, dataset, device, num_samples=3):
    """
    Визуализация предсказаний Faster R-CNN.
    Слева - Ground Truth, справа - Predictions.
    """
    model_rcnn.eval()
    class_names = ['background', 'without_mask', 'with_mask', 'mask_weared_incorrect']
    colors = ['gray', 'red', 'green', 'orange']
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(12, 4*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    with torch.no_grad():
        for i in range(num_samples):
            idx = random.randint(0, len(dataset) - 1)
            img, target = dataset[idx]
            img_np = img.permute(1, 2, 0).cpu().numpy()
            
            # Предсказание
            pred = model_rcnn([img.to(device)])[0]
            
            # Применяем NMS
            if len(pred['boxes']) > 0:
                keep = torchvision.ops.nms(pred['boxes'], pred['scores'], iou_threshold=0.3)
                pred['boxes'] = pred['boxes'][keep]
                pred['scores'] = pred['scores'][keep]
                pred['labels'] = pred['labels'][keep]
            
            # Ground Truth
            ax_gt = axes[i][0]
            ax_gt.imshow(img_np)
            ax_gt.set_title(f'Ground Truth (изображение {idx})')
            ax_gt.axis('off')
            
            for j, box in enumerate(target['boxes']):
                box = box.cpu().numpy()
                label_idx = int(target['labels'][j])
                rect = patches.Rectangle(
                    (box[0], box[1]), box[2]-box[0], box[3]-box[1],
                    linewidth=2, edgecolor=colors[label_idx], facecolor='none'
                )
                ax_gt.add_patch(rect)
                ax_gt.text(box[0], box[1]-5, class_names[label_idx], 
                          fontsize=8, color='white',
                          bbox=dict(facecolor=colors[label_idx], alpha=0.8))
            
            # Predictions
            ax_pred = axes[i][1]
            ax_pred.imshow(img_np)
            ax_pred.set_title(f'Faster R-CNN Predictions')
            ax_pred.axis('off')
            
            for j, box in enumerate(pred['boxes']):
                box = box.cpu().numpy()
                score = pred['scores'][j].item()
                label_idx = int(pred['labels'][j])
                rect = patches.Rectangle(
                    (box[0], box[1]), box[2]-box[0], box[3]-box[1],
                    linewidth=2, edgecolor=colors[label_idx], facecolor='none'
                )
                ax_pred.add_patch(rect)
                ax_pred.text(box[0], box[1]-5, f'{class_names[label_idx]} {score:.2f}',
                           fontsize=8, color='white',
                           bbox=dict(facecolor=colors[label_idx], alpha=0.8))
    
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'faster_rcnn_predictions.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Визуализация сохранена: {save_path}")

# Запуск визуализации
visualize_rcnn_predictions(model_rcnn, dataset_val, device, num_samples=3)


ВИЗУАЛИЗАЦИЯ ПРЕДСКАЗАНИЙ FASTER R-CNN


<Figure size 1200x1200 with 6 Axes>

Визуализация сохранена: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output\faster_rcnn_predictions.png


Визуализация предсказаний YOLO

In [33]:
print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ ПРЕДСКАЗАНИЙ YOLO")
print("=" * 60)

def visualize_yolo_predictions(model_yolo, yolo_dir, num_samples=3):
    """
    Визуализация предсказаний YOLO на случайных изображениях из валидации.
    """
    # Путь к валидационным изображениям
    val_img_dir = os.path.join(yolo_dir, 'images', 'val')
    
    if not os.path.exists(val_img_dir):
        print(f"Директория не найдена: {val_img_dir}")
        return
    
    # Получаем список изображений
    val_images = [os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if len(val_images) == 0:
        print("Изображения не найдены!")
        return
    
    # Выбираем случайные изображения
    sample_images = random.sample(val_images, min(num_samples, len(val_images)))
    
    # Делаем предсказания
    results = model_yolo.predict(sample_images, conf=0.25, save=False, verbose=False)
    
    # Визуализация
    fig, axes = plt.subplots(num_samples, 1, figsize=(10, 4*num_samples))
    if num_samples == 1:
        axes = [axes]
    
    class_names = ['without_mask', 'with_mask', 'mask_weared_incorrect']
    
    for i, (img_path, result) in enumerate(zip(sample_images, results)):
        # Загружаем изображение
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Создаем копию для рисования
        img_plot = img.copy()
        
        # Рисуем предсказанные боксы
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = box.conf[0].cpu().numpy()
                cls_id = int(box.cls[0].cpu().numpy())
                
                color = [(255, 0, 0), (0, 255, 0), (0, 165, 255)][cls_id]  # BGR для OpenCV
                # Конвертируем в RGB для matplotlib
                color_rgb = (color[2]/255, color[1]/255, color[0]/255)
                
                cv2.rectangle(img_plot, (x1, y1), (x2, y2), color, 2)
                label = f"{class_names[cls_id]} {conf:.2f}"
                cv2.putText(img_plot, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        axes[i].imshow(img_plot)
        axes[i].set_title(f'YOLO: {os.path.basename(img_path)}')
        axes[i].axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'yolo_predictions.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Визуализация сохранена: {save_path}")

# Запуск визуализации
visualize_yolo_predictions(model_yolo, yolo_dir, num_samples=3)


ВИЗУАЛИЗАЦИЯ ПРЕДСКАЗАНИЙ YOLO


<Figure size 1000x1200 with 3 Axes>

Визуализация сохранена: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output\yolo_predictions.png


Итоговое сравнение на одном изображении

In [34]:
print("\n" + "=" * 60)
print("СРАВНЕНИЕ МОДЕЛЕЙ НА ОДНОМ ИЗОБРАЖЕНИИ")
print("=" * 60)

# Берем одно изображение из валидационного датасета
idx = random.randint(0, len(dataset_val) - 1)
img, target = dataset_val[idx]
img_np = img.permute(1, 2, 0).cpu().numpy()

class_names = ['background', 'without_mask', 'with_mask', 'mask_weared_incorrect']
colors = ['gray', 'red', 'green', 'orange']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Ground Truth
axes[0].imshow(img_np)
axes[0].set_title('Ground Truth')
for j, box in enumerate(target['boxes']):
    box = box.cpu().numpy()
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor=colors[int(target['labels'][j])], facecolor='none')
    axes[0].add_patch(rect)
axes[0].axis('off')

# 2. Faster R-CNN Prediction
with torch.no_grad():
    pred_rcnn = model_rcnn([img.to(device)])[0]
    if len(pred_rcnn['boxes']) > 0:
        keep = torchvision.ops.nms(pred_rcnn['boxes'], pred_rcnn['scores'], 0.3)
        pred_rcnn['boxes'] = pred_rcnn['boxes'][keep]
        pred_rcnn['scores'] = pred_rcnn['scores'][keep]
        pred_rcnn['labels'] = pred_rcnn['labels'][keep]

axes[1].imshow(img_np)
axes[1].set_title('Faster R-CNN')
for j, box in enumerate(pred_rcnn['boxes']):
    box = box.cpu().numpy()
    score = pred_rcnn['scores'][j].item()
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor=colors[int(pred_rcnn['labels'][j])], facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(box[0], box[1]-5, f'{class_names[int(pred_rcnn["labels"][j])]} {score:.2f}',
                fontsize=8, color='white', bbox=dict(facecolor=colors[int(pred_rcnn['labels'][j])], alpha=0.8))
axes[1].axis('off')

# 3. YOLO Prediction
axes[2].imshow(img_np)
axes[2].set_title('YOLOv8')

# Ищем соответствующее изображение в YOLO датасете
img_name = None
for f in os.listdir(os.path.join(yolo_dir, 'images', 'val')):
    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_name = os.path.join(yolo_dir, 'images', 'val', f)
        break

if img_name:
    yolo_results = model_yolo.predict(img_name, conf=0.25, verbose=False)
    if yolo_results[0].boxes is not None:
        for box in yolo_results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = box.conf[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu().numpy())
            # Масштабируем координаты к размеру отображаемого изображения
            h_orig, w_orig = yolo_results[0].orig_img.shape[:2]
            scale_x = IMG_SIZE / w_orig
            scale_y = IMG_SIZE / h_orig
            x1_s = x1 * scale_x
            y1_s = y1 * scale_y
            x2_s = x2 * scale_x
            y2_s = y2 * scale_y
            rect = patches.Rectangle((x1_s, y1_s), x2_s-x1_s, y2_s-y1_s,
                                    linewidth=2, edgecolor=colors[cls_id+1], facecolor='none')
            axes[2].add_patch(rect)
            axes[2].text(x1_s, y1_s-5, f'{class_names[cls_id+1]} {conf:.2f}',
                        fontsize=8, color='white', bbox=dict(facecolor=colors[cls_id+1], alpha=0.8))
axes[2].axis('off')

plt.tight_layout()
save_path = os.path.join(output_dir, 'comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сравнение сохранено: {save_path}")


СРАВНЕНИЕ МОДЕЛЕЙ НА ОДНОМ ИЗОБРАЖЕНИИ


<Figure size 1800x600 with 3 Axes>

Сравнение сохранено: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output\comparison.png


In [35]:
summary = {
    'Faster_RCNN': {'mAP': float(mAP_rcnn), 'type': 'Two-stage', 'backbone': 'ResNet-50-FPN'},
    'YOLOv8n': {'mAP': float(metrics.box.map50), 'type': 'One-stage', 'backbone': 'CSPDarknet'}
}

with open(os.path.join(output_dir, 'results.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Результаты сохранены: {output_dir}/results.json")
print("Ноутбук завершен!")

Результаты сохранены: C:\Users\Public\Documents\Documents\Учебники\SF\CV\Projects\Project_2\output/results.json
Ноутбук завершен!
